# LEBER A4 - Label-Wise Dynamic Router - ResNet50 - BCE - Seed 52

A4 menguji inovasi inti LEBER: modul router dinamis adaptif independen untuk setiap label (w_L,c, w_R,c, w_B,c). Memungkinkan alokasi porsi berbeda untuk penyakit lokal (katarak) dan penyakit sistemik (diabetes/hipertensi) dengan penjaminan sifat exchange-equivariant. Backbone, loss, split, dan konfigurasi training identik dengan A0 s.d. A3.

In [1]:
from pathlib import Path
import pandas as pd

DATASET_DIR = Path(
    "/kaggle/input/datasets/kevinardhana/"
    "odir-5k-patient-level-multi-label-fundus-dataset"
)

IMAGE_DIR = DATASET_DIR / "Training Images" / "Training Images"

TRAIN_CSV = DATASET_DIR / "train.csv"
VALID_CSV = DATASET_DIR / "validation.csv"

LABELS = ["N", "D", "G", "C", "A", "H", "M", "O"]

train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)

print("Train      :", train_df.shape)
print("Validation :", valid_df.shape)
print("\nKolom:")
print(train_df.columns.tolist())

assert len(train_df) == 2450
assert len(valid_df) == 525
assert all(column in train_df.columns for column in LABELS)

print("\nContoh data:")
display(train_df.head())

print("\nCitra kiri contoh ada :", (IMAGE_DIR / train_df.loc[0, "left_image"]).is_file())
print("Citra kanan contoh ada:", (IMAGE_DIR / train_df.loc[0, "right_image"]).is_file())

Train      : (2450, 16)
Validation : (525, 16)

Kolom:
['patient_id', 'left_image', 'right_image', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O', 'label_count', 'is_multilabel', 'left_exists', 'right_exists', 'split']

Contoh data:


,patient_id,left_image,right_image,N,D,G,C,A,H,M,O,label_count,is_multilabel,left_exists,right_exists,split
0,1,1_left.jpg,1_right.jpg,1,0,0,0,0,0,0,0,1,0,True,True,train
1,3,3_left.jpg,3_right.jpg,0,0,0,0,0,0,0,1,1,0,True,True,train
2,4,4_left.jpg,4_right.jpg,0,1,0,0,0,0,0,1,2,1,True,True,train
3,5,5_left.jpg,5_right.jpg,0,1,0,0,0,0,0,0,1,0,True,True,train
4,6,6_left.jpg,6_right.jpg,0,1,0,0,0,0,0,1,2,1,True,True,train



Citra kiri contoh ada : True
Citra kanan contoh ada: True


In [2]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

for path in INPUT_ROOT.rglob("train.csv"):
    print("train.csv ditemukan di:", path)

for path in INPUT_ROOT.rglob("0_left.jpg"):
    print("Contoh citra ditemukan di:", path)

train.csv ditemukan di: /kaggle/input/datasets/kevinardhana/odir-5k-patient-level-multi-label-fundus-dataset/train.csv
Contoh citra ditemukan di: /kaggle/input/datasets/kevinardhana/odir-5k-patient-level-multi-label-fundus-dataset/Training Images/Training Images/0_left.jpg


In [3]:
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

SEED = 52
BATCH_SIZE = 16
NUM_WORKERS = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

train_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


class FundusPairDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        left_path = self.image_dir / row["left_image"]
        right_path = self.image_dir / row["right_image"]

        left_image = Image.open(left_path).convert("RGB")
        right_image = Image.open(right_path).convert("RGB")

        left_image = self.transform(left_image)
        right_image = self.transform(right_image)

        labels = torch.tensor(
            row[LABELS].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "left_image": left_image,
            "right_image": right_image,
            "labels": labels,
            "patient_id": int(row["patient_id"]),
        }


train_dataset = FundusPairDataset(train_df, IMAGE_DIR, train_transform)
valid_dataset = FundusPairDataset(valid_df, IMAGE_DIR, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


batch = next(iter(train_loader))

print("Citra kiri :", batch["left_image"].shape)
print("Citra kanan:", batch["right_image"].shape)
print("Label      :", batch["labels"].shape)
print("Contoh target:", batch["labels"][0].tolist())

Citra kiri : torch.Size([16, 3, 512, 512])
Citra kanan: torch.Size([16, 3, 512, 512])
Label      : torch.Size([16, 8])
Contoh target: [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0]


In [4]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


class LabelWiseRouterBilateralResNet50(nn.Module):
    """
    Ablasi A4: Tiga Expert (Kiri, Kanan, Bilateral) + Router Dinamis Per-Label (Label-Wise Routing).
    
    Menghasilkan bobot adaptif independen untuk setiap penyakit:
      w_{L, c} + w_{R, c} + w_{B, c} = 1.0, untuk c in {N, D, G, C, A, H, M, O}.
    
    Menjamin sifat matematis:
      - Exchange-Equivariant pada router: saat citra kiri dan kanan ditukar,
        bobot w_L dan w_R saling bertukar posisi untuk setiap label.
      - Exchange-Invariant pada prediksi: logit dan probabilitas pasien bernilai identik (Delta p = 0.0000).
    """
    def __init__(self, num_labels=8, dropout=0.30, pretrained=True):
        super().__init__()
        self.num_labels = num_labels

        weights = (
            ResNet50_Weights.IMAGENET1K_V2
            if pretrained else None
        )
        self.backbone = resnet50(weights=weights)

        feature_dim = self.backbone.fc.in_features  # 2048
        self.backbone.fc = nn.Identity()

        # Shared Monocular Expert: Linear(2048 -> 8) [equivariant monocular branch]
        self.expert_mono = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(feature_dim, num_labels)
        )

        # Bilateral Expert: Linear(6144 -> 8)
        self.expert_bilateral = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(feature_dim * 3, num_labels)
        )

        # Equivariant Label-Wise Router
        # Menghasilkan skor routing berdimensi [B, num_labels] untuk masing-masing cabang
        self.router_mono = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_labels)
        )
        self.router_bilateral = nn.Sequential(
            nn.Linear(feature_dim * 3, 64),
            nn.ReLU(),
            nn.Linear(64, num_labels)
        )

    @staticmethod
    def symmetric_features(left_features, right_features):
        sum_features = left_features + right_features
        difference_features = torch.abs(left_features - right_features)
        product_features = left_features * right_features
        return torch.cat(
            [sum_features, difference_features, product_features],
            dim=1
        )

    def forward(self, left_image, right_image, return_weights=False):
        left_features = self.backbone(left_image)
        right_features = self.backbone(right_image)

        # Pendapat awal masing-masing expert: [B, num_labels]
        z_left = self.expert_mono(left_features)
        z_right = self.expert_mono(right_features)

        bilateral_features = self.symmetric_features(
            left_features,
            right_features
        )
        z_bilateral = self.expert_bilateral(bilateral_features)

        # Scoring router per-label: masing-masing berukuran [B, num_labels]
        s_left = self.router_mono(left_features)
        s_right = self.router_mono(right_features)
        s_bilateral = self.router_bilateral(bilateral_features)

        # Stack sepanjang dimensi cabang (dim=1) -> [B, 3, num_labels]
        scores = torch.stack([s_left, s_right, s_bilateral], dim=1)

        # Softmax sepanjang dimensi cabang (dim=1) secara independen untuk tiap label
        weights = torch.softmax(scores, dim=1)  # [B, 3, num_labels]

        w_left = weights[:, 0, :]       # [B, num_labels]
        w_right = weights[:, 1, :]      # [B, num_labels]
        w_bilateral = weights[:, 2, :]  # [B, num_labels]

        # Fusi bukti tertimbang per-label: [B, num_labels]
        # (w_left * z_left + w_right * z_right) bersifat komutatif terhadap pertukaran kiri-kanan
        fused_logits = (w_left * z_left + w_right * z_right) + (w_bilateral * z_bilateral)

        if return_weights:
            return fused_logits, weights
        return fused_logits


model = LabelWiseRouterBilateralResNet50(
    num_labels=len(LABELS)
).to(DEVICE)

left_batch = batch["left_image"].to(DEVICE)
right_batch = batch["right_image"].to(DEVICE)

model.eval()
with torch.no_grad():
    logits, weights_orig = model(left_batch, right_batch, return_weights=True)
    swapped_logits, weights_swap = model(right_batch, left_batch, return_weights=True)
    probabilities = torch.sigmoid(logits)

swap_error_p = torch.max(
    torch.abs(logits - swapped_logits)
).item()

# Cek equivariance per label: w_left <-> w_right, w_bilat invariant
swap_error_w_mono = torch.max(
    torch.abs(weights_orig[:, 0, :] - weights_swap[:, 1, :])
).item()

swap_error_w_bilat = torch.max(
    torch.abs(weights_orig[:, 2, :] - weights_swap[:, 2, :])
).item()

assert logits.shape == (BATCH_SIZE, len(LABELS))
assert weights_orig.shape == (BATCH_SIZE, 3, len(LABELS))
assert swap_error_p < 1e-6, f"Prediksi A4 tidak invariant terhadap swap: {swap_error_p}"
assert swap_error_w_mono < 1e-6, f"Bobot monocular A4 tidak equivariant: {swap_error_w_mono}"
assert swap_error_w_bilat < 1e-6, f"Bobot bilateral A4 tidak invariant: {swap_error_w_bilat}"

print("Logits shape       :", logits.shape)
print("Weights shape      :", weights_orig.shape)
print("Maksimum selisih swap logit (Delta p):", swap_error_p)
print("Maksimum selisih swap bobot (Delta w):", max(swap_error_w_mono, swap_error_w_bilat))
print(
    "Parameter trainable total:",
    sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
)
print(
    "Parameter classifier 3 Expert:",
    sum(p.numel() for p in model.expert_mono.parameters() if p.requires_grad) +
    sum(p.numel() for p in model.expert_bilateral.parameters() if p.requires_grad)
)
print(
    "Parameter Label-Wise Router MLP:",
    sum(p.numel() for p in model.router_mono.parameters() if p.requires_grad) +
    sum(p.numel() for p in model.router_bilateral.parameters() if p.requires_grad)
)

Device: cuda
GPU: Tesla T4
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 230MB/s]


Logits shape       : torch.Size([16, 8])
Weights shape      : torch.Size([16, 3, 8])
Maksimum selisih swap logit (Delta p): 0.0
Maksimum selisih swap bobot (Delta w): 0.0
Parameter trainable total: 24099040
Parameter classifier 3 Expert: 65552
Parameter Label-Wise Router MLP: 525456


In [5]:
import json
from pathlib import Path
from sklearn.metrics import f1_score
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 7

OUTPUT_DIR = Path("/kaggle/working/leber_a4_labelwise_router_bce_512_seed52")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for batch in loader:
        left_images = batch["left_image"].to(device)
        right_images = batch["right_image"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=device.type == "cuda"
        ):
            logits = model(left_images, right_images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    all_targets = []
    all_probabilities = []

    for batch in loader:
        left_images = batch["left_image"].to(device)
        right_images = batch["right_image"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=device.type == "cuda"
        ):
            logits = model(left_images, right_images)
            loss = criterion(logits, labels)
        probabilities = torch.sigmoid(logits.float())

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_targets.append(labels.cpu().numpy())
        all_probabilities.append(probabilities.cpu().numpy())

    y_true = np.concatenate(all_targets)
    y_prob = np.concatenate(all_probabilities)
    y_pred = (y_prob >= threshold).astype(int)

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    return {
        "loss": total_loss / total_samples,
        "macro_f1": macro_f1,
        "y_true": y_true,
        "y_prob": y_prob,
    }


print("BCE, AdamW, scheduler, dan fungsi training siap.")

BCE, AdamW, scheduler, dan fungsi training siap.


In [6]:
history = []
best_macro_f1 = -1.0
best_epoch = 0
patience_counter = 0

checkpoint_path = OUTPUT_DIR / "best_leber_a4_labelwise_router_resnet50_bce_512_seed52.pt"
history_path = OUTPUT_DIR / "training_history.json"

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=DEVICE
    )

    validation_result = evaluate(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=DEVICE,
        threshold=0.5
    )

    validation_loss = validation_result["loss"]
    validation_macro_f1 = validation_result["macro_f1"]

    scheduler.step(validation_macro_f1)

    current_lr = optimizer.param_groups[0]["lr"]

    epoch_result = {
        "epoch": epoch,
        "train_loss": float(train_loss),
        "validation_loss": float(validation_loss),
        "validation_macro_f1": float(validation_macro_f1),
        "learning_rate": float(current_lr),
    }
    history.append(epoch_result)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train loss: {train_loss:.4f} | "
        f"val loss: {validation_loss:.4f} | "
        f"val Macro-F1: {validation_macro_f1:.4f} | "
        f"lr: {current_lr:.6f}"
    )

    if validation_macro_f1 > best_macro_f1:
        best_macro_f1 = validation_macro_f1
        best_epoch = epoch
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "validation_macro_f1": best_macro_f1,
                "labels": LABELS,
                "threshold": 0.5,
                "backbone": "shared_resnet50_labelwise_router",
                "input_size": 512,
                "batch_size": 16,
                "seed": 52,
                "loss_function": "BCEWithLogitsLoss",
            },
            checkpoint_path
        )

        print("  âœ“ Checkpoint terbaik disimpan.")

    else:
        patience_counter += 1

    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(
            f"Early stopping pada epoch {epoch}. "
            f"Checkpoint terbaik: epoch {best_epoch}."
        )
        break


with history_path.open("w") as file:
    json.dump(history, file, indent=2)

print("\nPelatihan selesai.")
print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", round(best_macro_f1, 4))
print("Checkpoint:", checkpoint_path)

Epoch 01/30 | train loss: 0.3392 | val loss: 0.2947 | val Macro-F1: 0.2596 | lr: 0.000100
  âœ“ Checkpoint terbaik disimpan.
Epoch 02/30 | train loss: 0.2713 | val loss: 0.2895 | val Macro-F1: 0.4087 | lr: 0.000100
  âœ“ Checkpoint terbaik disimpan.
Epoch 03/30 | train loss: 0.2382 | val loss: 0.2657 | val Macro-F1: 0.4941 | lr: 0.000100
  âœ“ Checkpoint terbaik disimpan.
Epoch 04/30 | train loss: 0.2000 | val loss: 0.2623 | val Macro-F1: 0.4568 | lr: 0.000100
Epoch 05/30 | train loss: 0.1684 | val loss: 0.2743 | val Macro-F1: 0.5153 | lr: 0.000100
  âœ“ Checkpoint terbaik disimpan.
Epoch 06/30 | train loss: 0.1332 | val loss: 0.2680 | val Macro-F1: 0.5279 | lr: 0.000100
  âœ“ Checkpoint terbaik disimpan.
Epoch 07/30 | train loss: 0.1055 | val loss: 0.3394 | val Macro-F1: 0.5097 | lr: 0.000100
Epoch 08/30 | train loss: 0.1000 | val loss: 0.3046 | val Macro-F1: 0.4938 | lr: 0.000100
Epoch 09/30 | train loss: 0.0742 | val loss: 0.3410 | val Macro-F1: 0.6084 | lr: 0.000100
  âœ“ Checkpoin

In [7]:
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/leber_a4_labelwise_router_bce_512_seed52")
checkpoint_path = OUTPUT_DIR / "best_leber_a4_labelwise_router_resnet50_bce_512_seed52.pt"

print("Lokasi checkpoint:", checkpoint_path)
print("Checkpoint tersedia:", checkpoint_path.exists())

Lokasi checkpoint: /kaggle/working/leber_a4_labelwise_router_bce_512_seed52/best_leber_a4_labelwise_router_resnet50_bce_512_seed52.pt
Checkpoint tersedia: True


In [8]:
# Memuat checkpoint terbaik dan menentukan threshold BCE
# menggunakan validation set

from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import resnet50


LABELS = ["N", "D", "G", "C", "A", "H", "M", "O"]

BATCH_SIZE = 16
NUM_WORKERS = 2

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# Lokasi dataset dan hasil training
DATASET_DIR = Path(
    "/kaggle/input/datasets/kevinardhana/"
    "odir-5k-patient-level-multi-label-fundus-dataset"
)

IMAGE_DIR = (
    DATASET_DIR
    / "Training Images"
    / "Training Images"
)

VALID_CSV = DATASET_DIR / "validation.csv"

OUTPUT_DIR = Path(
    "/kaggle/working/leber_a4_labelwise_router_bce_512_seed52"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Dataset pasangan citra mata kiri dan kanan
class FundusPairDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_dir,
        transform
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        left_path = (
            self.image_dir
            / row["left_image"]
        )

        right_path = (
            self.image_dir
            / row["right_image"]
        )

        left_image = Image.open(
            left_path
        ).convert("RGB")

        right_image = Image.open(
            right_path
        ).convert("RGB")

        left_image = self.transform(
            left_image
        )

        right_image = self.transform(
            right_image
        )

        labels = torch.tensor(
            row[LABELS].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "left_image": left_image,
            "right_image": right_image,
            "labels": labels
        }


# Arsitektur harus sama dengan model saat training
class LabelWiseRouterBilateralResNet50(nn.Module):
    def __init__(self, num_labels=8, dropout=0.30):
        super().__init__()
        self.num_labels = num_labels
        self.backbone = resnet50(weights=None)
        feature_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.expert_mono = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(feature_dim, num_labels)
        )
        self.expert_bilateral = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(feature_dim * 3, num_labels)
        )

        self.router_mono = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_labels)
        )
        self.router_bilateral = nn.Sequential(
            nn.Linear(feature_dim * 3, 64),
            nn.ReLU(),
            nn.Linear(64, num_labels)
        )

    @staticmethod
    def symmetric_features(left_features, right_features):
        return torch.cat(
            [
                left_features + right_features,
                torch.abs(left_features - right_features),
                left_features * right_features
            ],
            dim=1
        )

    def forward(self, left_image, right_image, return_weights=False):
        left_features = self.backbone(left_image)
        right_features = self.backbone(right_image)

        z_left = self.expert_mono(left_features)
        z_right = self.expert_mono(right_features)

        bilateral_features = self.symmetric_features(
            left_features,
            right_features
        )
        z_bilateral = self.expert_bilateral(bilateral_features)

        s_left = self.router_mono(left_features)
        s_right = self.router_mono(right_features)
        s_bilateral = self.router_bilateral(bilateral_features)

        scores = torch.stack([s_left, s_right, s_bilateral], dim=1)
        weights = torch.softmax(scores, dim=1)  # [B, 3, 8]

        w_left = weights[:, 0, :]
        w_right = weights[:, 1, :]
        w_bilateral = weights[:, 2, :]

        fused_logits = (w_left * z_left + w_right * z_right) + (w_bilateral * z_bilateral)
        if return_weights:
            return fused_logits, weights
        return fused_logits


@torch.no_grad()
def collect_predictions(
    model,
    loader,
    device
):
    model.eval()

    all_targets = []
    all_probabilities = []

    for batch in loader:
        left_images = batch["left_image"].to(
            device,
            non_blocking=True
        )

        right_images = batch["right_image"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        logits = model(
            left_images,
            right_images
        )

        probabilities = torch.sigmoid(
            logits
        )

        all_targets.append(
            labels.cpu().numpy()
        )

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

    targets = np.concatenate(
        all_targets,
        axis=0
    )

    probabilities = np.concatenate(
        all_probabilities,
        axis=0
    )

    return targets, probabilities


# Transformasi validation sama dengan baseline
eval_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Membaca validation set
valid_df = pd.read_csv(
    VALID_CSV
)

assert len(valid_df) == 525, (
    f"Jumlah validation tidak sesuai: {len(valid_df)}"
)

assert all(
    label in valid_df.columns
    for label in LABELS
), "Kolom label validation tidak lengkap"


valid_dataset = FundusPairDataset(
    dataframe=valid_df,
    image_dir=IMAGE_DIR,
    transform=eval_transform
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda"
)


# Checkpoint dibuat oleh cell training sebelumnya
checkpoint_path = (
    OUTPUT_DIR
    / "best_leber_a4_labelwise_router_resnet50_bce_512_seed52.pt"
)

assert checkpoint_path.is_file(), (
    "Checkpoint tidak ditemukan. "
    f"Lokasi yang diperiksa: {checkpoint_path}"
)


checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)


required_checkpoint_keys = {
    "epoch",
    "model_state_dict"
}

missing_keys = (
    required_checkpoint_keys
    - set(checkpoint.keys())
)

assert not missing_keys, (
    "Isi checkpoint tidak lengkap. "
    f"Key yang tidak ditemukan: {missing_keys}"
)


# Membuat model dan memuat bobot terbaik
model = LabelWiseRouterBilateralResNet50(
    num_labels=len(LABELS),
    dropout=0.30
).to(DEVICE)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()


print("Checkpoint ditemukan:", checkpoint_path)
print("Checkpoint epoch:", checkpoint["epoch"])
print("Device evaluasi:", DEVICE)


# Menghasilkan probabilitas validation
y_val, prob_val = collect_predictions(
    model=model,
    loader=valid_loader,
    device=DEVICE
)


assert y_val.shape == (525, 8), (
    f"Bentuk target validation tidak sesuai: {y_val.shape}"
)

assert prob_val.shape == (525, 8), (
    "Bentuk probabilitas validation "
    f"tidak sesuai: {prob_val.shape}"
)


# Mencari threshold terbaik untuk setiap label
candidate_thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

best_thresholds = []
threshold_rows = []


for label_index, label_name in enumerate(LABELS):
    label_scores = []

    for threshold in candidate_thresholds:
        label_predictions = (
            prob_val[:, label_index]
            >= threshold
        ).astype(np.int32)

        score = f1_score(
            y_val[:, label_index],
            label_predictions,
            zero_division=0
        )

        label_scores.append(score)

    best_position = int(
        np.argmax(label_scores)
    )

    selected_threshold = float(
        candidate_thresholds[best_position]
    )

    selected_f1 = float(
        label_scores[best_position]
    )

    best_thresholds.append(
        selected_threshold
    )

    threshold_rows.append({
        "label": label_name,
        "threshold_terbaik":
            selected_threshold,
        "F1_validation":
            selected_f1,
        "jumlah_positif_validation":
            int(y_val[:, label_index].sum())
    })


best_thresholds = np.asarray(
    best_thresholds,
    dtype=np.float32
)


# Membandingkan threshold 0,50 dengan threshold per label
default_predictions = (
    prob_val >= 0.50
).astype(np.int32)

optimized_predictions = (
    prob_val >= best_thresholds
).astype(np.int32)


macro_f1_default = f1_score(
    y_val,
    default_predictions,
    average="macro",
    zero_division=0
)

macro_f1_optimized = f1_score(
    y_val,
    optimized_predictions,
    average="macro",
    zero_division=0
)


threshold_df = pd.DataFrame(
    threshold_rows
)


print(
    "\nMacro-F1 threshold 0,50:",
    f"{macro_f1_default:.4f}"
)

print(
    "Macro-F1 threshold per label:",
    f"{macro_f1_optimized:.4f}"
)

print("\nThreshold setiap label:")
display(threshold_df)


# Menyimpan threshold
threshold_csv_path = (
    OUTPUT_DIR
    / "validation_thresholds_leber_a4_labelwise_router_bce_512_seed52.csv"
)

threshold_json_path = (
    OUTPUT_DIR
    / "validation_thresholds_leber_a4_labelwise_router_bce_512_seed52.json"
)


threshold_df.to_csv(
    threshold_csv_path,
    index=False
)


with open(
    threshold_json_path,
    "w"
) as file:
    json.dump(
        {
            "checkpoint_epoch":
                int(checkpoint["epoch"]),
            "labels":
                LABELS,
            "thresholds":
                best_thresholds.tolist(),
            "macro_f1_threshold_0_5":
                float(macro_f1_default),
            "macro_f1_optimized":
                float(macro_f1_optimized)
        },
        file,
        indent=2
    )


print("\nThreshold berhasil disimpan:")
print("-", threshold_csv_path)
print("-", threshold_json_path)

Checkpoint ditemukan: /kaggle/working/leber_a4_labelwise_router_bce_512_seed52/best_leber_a4_labelwise_router_resnet50_bce_512_seed52.pt
Checkpoint epoch: 14
Device evaluasi: cuda

Macro-F1 threshold 0,50: 0.6186
Macro-F1 threshold per label: 0.6481

Threshold setiap label:


,label,threshold_terbaik,F1_validation,jumlah_positif_validation
0,N,0.05,0.664879,171
1,D,0.48,0.700337,169
2,G,0.92,0.538462,32
3,C,0.66,0.857143,32
4,A,0.23,0.607143,25
5,H,0.55,0.347826,16
6,M,0.57,0.897959,26
7,O,0.37,0.571429,147



Threshold berhasil disimpan:
- /kaggle/working/leber_a4_labelwise_router_bce_512_seed52/validation_thresholds_leber_a4_labelwise_router_bce_512_seed52.csv
- /kaggle/working/leber_a4_labelwise_router_bce_512_seed52/validation_thresholds_leber_a4_labelwise_router_bce_512_seed52.json


In [9]:
# Audit konsistensi pertukaran mata pada validation set (LEBER A4)
# Mengukur Delta p (probabilitas), Delta w (bobot router), dan analisis distribusi bobot per label

@torch.no_grad()
def collect_original_and_swapped_outputs(model, loader, device):
    model.eval()
    all_targets = []
    original_probabilities = []
    swapped_probabilities = []
    original_weights_list = []
    swapped_weights_list = []

    for batch in loader:
        left_images = batch["left_image"].to(device)
        right_images = batch["right_image"].to(device)
        labels = batch["labels"].cpu().numpy()

        orig_logits, orig_w = model(left_images, right_images, return_weights=True)
        swap_logits, swap_w = model(right_images, left_images, return_weights=True)

        all_targets.append(labels)
        original_probabilities.append(
            torch.sigmoid(orig_logits).cpu().numpy()
        )
        swapped_probabilities.append(
            torch.sigmoid(swap_logits).cpu().numpy()
        )
        original_weights_list.append(orig_w.cpu().numpy())
        swapped_weights_list.append(swap_w.cpu().numpy())

    return (
        np.concatenate(all_targets),
        np.concatenate(original_probabilities),
        np.concatenate(swapped_probabilities),
        np.concatenate(original_weights_list),
        np.concatenate(swapped_weights_list)
    )


y_val, prob_original, prob_swapped, weights_original, weights_swapped = (
    collect_original_and_swapped_outputs(
        model=model,
        loader=valid_loader,
        device=DEVICE
    )
)

absolute_probability_diff = np.abs(prob_original - prob_swapped)

# Evaluasi bobot router per-label:
# weights_original: [N, 3, 8], weights_swapped: [N, 3, 8]
# branch 0 (kiri) orig harus sama dengan branch 1 (kanan) swap
# branch 1 (kanan) orig harus sama dengan branch 0 (kiri) swap
# branch 2 (bilat) orig harus sama dengan branch 2 (bilat) swap
delta_w_left = np.abs(weights_original[:, 0, :] - weights_swapped[:, 1, :])
delta_w_right = np.abs(weights_original[:, 1, :] - weights_swapped[:, 0, :])
delta_w_bilat = np.abs(weights_original[:, 2, :] - weights_swapped[:, 2, :])
max_delta_w = float(max(delta_w_left.max(), delta_w_right.max(), delta_w_bilat.max()))

predictions_orig = (prob_original >= best_thresholds).astype(np.int32)
predictions_swap = (prob_swapped >= best_thresholds).astype(np.int32)

swap_metrics = {
    "split": "validation",
    "mean_absolute_probability_difference": float(absolute_probability_diff.mean()),
    "maximum_absolute_probability_difference": float(absolute_probability_diff.max()),
    "maximum_router_weight_difference": max_delta_w,
    "label_decision_disagreement_rate": float(
        np.not_equal(predictions_orig, predictions_swap).mean()
    ),
    "patient_exact_agreement_rate": float(
        np.all(predictions_orig == predictions_swap, axis=1).mean()
    )
}

assert swap_metrics["mean_absolute_probability_difference"] < 1e-6
assert swap_metrics["maximum_router_weight_difference"] < 1e-6
assert swap_metrics["label_decision_disagreement_rate"] == 0.0

swap_metrics_path = (
    OUTPUT_DIR
    / "validation_swap_metrics_leber_a4_labelwise_router_bce_512_seed52.json"
)

with open(swap_metrics_path, "w") as file:
    json.dump(swap_metrics, file, indent=2)

print("Audit swap validation LEBER A4:")
for key, value in swap_metrics.items():
    print(f"{key}: {value}")

# Analisis Distribusi Bobot Router per Label Penyakit
# Rata-rata bobot w_L, w_R, w_B untuk setiap dari 8 label
LABEL_NAMES = {
    "N": "Normal",
    "D": "Diabetes",
    "G": "Glaucoma",
    "C": "Cataract",
    "A": "AMD",
    "H": "Hypertension",
    "M": "Myopia",
    "O": "Others"
}

labelwise_weights_rows = []
print("\nDistribusi Rata-Rata Bobot Router per Label:")
print(f"{'Label':<6} {'Nama Penyakit':<20} {'w_Left':<12} {'w_Right':<12} {'w_Bilateral':<12}")
print("-" * 65)

for idx, label in enumerate(LABELS):
    mean_w_l = float(weights_original[:, 0, idx].mean())
    mean_w_r = float(weights_original[:, 1, idx].mean())
    mean_w_b = float(weights_original[:, 2, idx].mean())
    
    labelwise_weights_rows.append({
        "label": label,
        "mean_weight_left": mean_w_l,
        "mean_weight_right": mean_w_r,
        "mean_weight_bilateral": mean_w_b
    })
    print(f"{label:<6} {LABEL_NAMES[label]:<20} {mean_w_l:<12.4f} {mean_w_r:<12.4f} {mean_w_b:<12.4f}")

weights_summary_df = pd.DataFrame(labelwise_weights_rows)
weights_summary_path = (
    OUTPUT_DIR
    / "validation_labelwise_weights_summary_leber_a4_bce_512_seed52.csv"
)
weights_summary_df.to_csv(weights_summary_path, index=False)
print(f"\nRingkasan bobot per-label disimpan di: {weights_summary_path}")


# Simpan probabilitas validasi untuk evaluasi ensemble multi-seed
np.save(OUTPUT_DIR / "validation_probabilities_leber_a4_bce_512_seed52.npy", prob_original)
np.save(OUTPUT_DIR / "validation_targets_leber_a4_bce_512_seed52.npy", y_val)
print("Probabilitas validasi berhasil disimpan untuk ensemble multi-seed.")

print("\nArtefak disimpan di:", OUTPUT_DIR)
print("Test set tetap terjaga bebas dari kebocoran data untuk evaluasi final.")

Audit swap validation LEBER A4:
split: validation
mean_absolute_probability_difference: 0.0
maximum_absolute_probability_difference: 0.0
maximum_router_weight_difference: 0.0
label_decision_disagreement_rate: 0.0
patient_exact_agreement_rate: 1.0

Distribusi Rata-Rata Bobot Router per Label:
Label  Nama Penyakit        w_Left       w_Right      w_Bilateral 
-----------------------------------------------------------------
N      Normal               0.0833       0.0865       0.8302      
D      Diabetes             0.2030       0.1790       0.6179      
G      Glaucoma             0.0003       0.0003       0.9995      
C      Cataract             0.0624       0.0693       0.8683      
A      AMD                  0.0005       0.0005       0.9991      
H      Hypertension         0.0852       0.0887       0.8260      
M      Myopia               0.0321       0.0391       0.9289      
O      Others               0.0100       0.0100       0.9801      

Ringkasan bobot per-label disimpan di